In [1]:
import os
from dotenv import load_dotenv

load_dotenv(r"../backend/.env")

True

In [2]:
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USER=os.getenv("NEO4J_USER")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")
NEO4J_DB="graphacademy"

AWS_REGION=os.getenv("AWS_REGION")
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")

AWS_BEDROCK_HAIKU_ARN=os.getenv("AWS_BEDROCK_HAIKU_ARN")
AWS_BEDROCK_COHERE_EMBED_V4_ARN=os.getenv("AWS_BEDROCK_COHERE_EMBED_V4_ARN")

In [3]:
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.config import Neo4jConfig
from neo4j_agent_memory.config import Neo4jConfig, EmbeddingConfig, LLMConfig, ExtractionConfig, ExtractorType  
from  neo4j_agent_memory.extraction import ExtractorBuilder

In [4]:
from neo4j_agent_memory.llm.adapters.litellm import LiteLLMProvider, LiteLLMEmbeddingProvider  
  
settings = MemorySettings(  
    neo4j=Neo4jConfig(
        uri=NEO4J_URI,
        username=NEO4J_USER,
        password=NEO4J_PASSWORD,
        database=NEO4J_DB,
    ),  
    llm=LiteLLMProvider(  
        model=f"bedrock/{AWS_BEDROCK_HAIKU_ARN}",   
        api_key=AWS_BEARER_TOKEN_BEDROCK,
        aws_region=AWS_REGION,  
    ),  
    embedding=LiteLLMEmbeddingProvider(  
        model=f"bedrock/{AWS_BEDROCK_COHERE_EMBED_V4_ARN}",  
        api_key=AWS_BEARER_TOKEN_BEDROCK,  
        aws_region=AWS_REGION,  
        dimensions=1536,  
    ),
    extraction=ExtractionConfig(  
        extractor_type=ExtractorType.NONE,  # No spaCy or GLiNER
        enable_spacy=False,
        enable_gliner=False,
        enable_llm_fallback=False,
    ),  
)

In [5]:
extractor = ExtractorBuilder().with_spacy(model="en_core_web_lg").with_gliner(model="gliner-community/gliner_large-v2.5").merge_by_confidence().build()

In [6]:
user_message = """Show me the breakdown of cross-border payment volume by corridor, currency, and method for the past 12 months, with end-to-end cost including FX spread, lifting fees, and correspondent charges."""


asst_message = """"Data Quality Concern
Two data integrity issues compromise this analysis and must be resolved before any cost findings are acted upon.

Invalid origination country code 'HO' appears across multiple rows. ISO 3166-1 alpha-2 does not recognize this code — corridor routing and origination country segmentation are unreliable wherever this code appears, meaning any corridor-level cost comparison may be misattributed.

Future-dated records: the dataset includes transactions dated August 2026, which is 58 days ahead of today (2026-06-09). Whether these represent forecasted, test, or misdated records is unknown — their inclusion in volume and cost aggregates cannot be validated until the source is confirmed.

All findings below are framed pending resolution of these concerns.

Verdict
The end-to-end cost breakdown you requested cannot be delivered: correspondent charges and total cost figures are entirely absent across all 2,495,075 transactions — and at industry-standard rates, this blind spot likely understates true payment friction by $100M+ annually.

What This Means
Critical cost data is missing, not just incomplete — correspondent_charges_amount and end_to_end_cost_monetary are NULL across every one of the 2,495,075 rows in the dataset. Correspondent charges typically represent 15–25% of total cross-border payment cost; omitting them means every cost-per-corridor figure currently in use is structurally understated. The $100M+ annual understatement is an industry-standard estimate applied to this dataset — the actual figure cannot be confirmed until the fields are populated.
Lifting fee variability signals an unmanaged cost spread — fees range from $16.20 to $4,219.13 per transaction, with a mean of $2,780.53 and median of $3,235.29. The Turkey-to-Hong Kong corridor carries the highest observed fees at $3,723.37, versus $2,650.30 for Korea-to-Singapore — a $1,073 per-transaction gap that, at scale, represents a material and likely unoptimized cost differential across corridors.

FX spread adds further uneven cost pressure — spreads range from 33.0 to 79.0 basis points, with a mean of 51.68 bps. The 46 bps spread between the tightest and widest corridor is not a rounding difference; on large-value cross-border flows, this range determines whether a corridor is competitively priced or quietly expensive.

Decision
Direct Data Engineering to identify why correspondent_charges_amount and end_to_end_cost_monetary are NULL across all 2,495,075 rows, and restore or source these fields within 10 business days.

If actioned: full end-to-end cost visibility is restored across the entire transaction population, enabling defensible corridor-level cost benchmarking and fee negotiation with correspondent banks.

If deferred: every corridor cost decision — pricing, routing, and bank relationship reviews — continues to be made against an incomplete cost base that structurally understates true payment friction, with the $100M+ annual gap remaining invisible to management."""

In [9]:
result = await extractor.extract(user_message)

In [ ]:
result2 = await extractor.extract(asst_message)

: 

In [ ]:
result2

In [ ]:
result2.model_dump()

In [ ]:
result.model_dump()

In [ ]:
import time

async with MemoryClient(settings) as memory:
    t0 = time.perf_counter()
    await memory.short_term.add_message(
        session_id="user_123",
        role="user",
        extract_entities=False,
        extract_relations=False,
        content=user_message
    )
    t1 = time.perf_counter()
    print(f"add_message (user):      {t1 - t0:.3f}s")

    await memory.short_term.add_message(
        session_id="user_123",
        role="assistant",
        extract_entities=False,
        extract_relations=False,
        content=asst_message
    )
    t2 = time.perf_counter()
    print(f"add_message (assistant): {t2 - t1:.3f}s")

    conversation = await memory.short_term.get_conversation("user_123")
    t3 = time.perf_counter()
    print(f"get_conversation:        {t3 - t2:.3f}s")

    results = await memory.short_term.search_messages(
        query="cross border payment cost breakdown",
        session_id="user_123",
        limit=10
    )
    t4 = time.perf_counter()
    print(f"search_messages:         {t4 - t3:.3f}s")

    print(f"\n--- Total: {t4 - t0:.3f}s ---")

    # Print without embedding vectors
    for msg in conversation.messages:
        print(f"[{msg.role}] {msg.content[:80]}...")
    print(f"\nSearch results: {len(results)} message(s) found")

In [ ]:
----

In [ ]:
# Isolate Bedrock embedding latency
import time
from neo4j_agent_memory.llm.adapters.litellm import LiteLLMEmbeddingProvider

emb = LiteLLMEmbeddingProvider(
    model=f"bedrock/{AWS_BEDROCK_COHERE_EMBED_V4_ARN}",
    api_key=AWS_BEARER_TOKEN_BEDROCK,
    aws_region=AWS_REGION,
    dimensions=1536,
)

t0 = time.perf_counter()
out_1 = await emb.embed([user_message])
t1 = time.perf_counter()
print(f"Embed user message:      {t1 - t0:.3f}s")

out_2 = await emb.embed([asst_message])
t2 = time.perf_counter()
print(f"Embed assistant message: {t2 - t1:.3f}s")

In [ ]:
# Isolate Neo4j write latency directly
import time
from neo4j import AsyncGraphDatabase

driver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

async with driver.session(database=NEO4J_DB) as session:
    t0 = time.perf_counter()
    await session.run("CREATE (n:TestNode {msg: $msg}) RETURN n", msg=user_message)
    t1 = time.perf_counter()
    print(f"Neo4j write (user msg):      {t1 - t0:.3f}s")

    await session.run("CREATE (n:TestNode {msg: $msg}) RETURN n", msg=asst_message)
    t2 = time.perf_counter()
    print(f"Neo4j write (assistant msg): {t2 - t1:.3f}s")

    # Cleanup
    await session.run("MATCH (n:TestNode) DELETE n")

await driver.close()

print("\nIf Neo4j writes are fast (<1s), the bottleneck is spaCy running inside add_message.")